In [32]:
import pandas as pd
from surprise import Dataset, Reader
from time import perf_counter
from surprise import SVD, KNNBasic, accuracy
from surprise.model_selection import train_test_split, GridSearchCV, KFold

# Loading the dataset

In [33]:
data = pd.read_csv(r"ml-100k.csv")
data.head()

,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [34]:
surprise_data = Dataset.load_from_df(
    data[["user_id", "item_id", "rating"]],
    Reader(rating_scale=(1, 5)),
)

# Train - Test split

In [35]:
trainset, testset = train_test_split(
    surprise_data, test_size=0.25, random_state=42
)

# Task 1:
Try the proposed factors in different SVD models, and observe how increasing the number of latent factors affects model accuracy(RMSE).

factors_to_test = [2, 5, 10, 20]

In [36]:
factors_to_test = [2, 5, 10, 20]
factor_results = []

for n_factors in factors_to_test:
    model = SVD(n_factors=n_factors, random_state=42)
    model.fit(trainset)
    predictions = model.test(testset)
    factor_results.append({
        "n_factors": n_factors,
        "RMSE": accuracy.rmse(predictions, verbose=False),
    })

factor_results = pd.DataFrame(factor_results)
display(factor_results)
best_factor_count = int(factor_results.loc[factor_results["RMSE"].idxmin(), "n_factors"])
print(f"Lowest test RMSE among these models: n_factors={best_factor_count}")

,n_factors,RMSE
0,2,0.942582
1,5,0.940043
2,10,0.938027
3,20,0.939483


Lowest test RMSE among these models: n_factors=10


# Task 2:
Generate Top-5 for the user with the ID: 196.

For recommendations, refit the selected SVD model on all known ratings and predict only movies this user has not rated. This model is separate from the models used for test-set evaluation. The CSV contains numeric IDs, so use `196`, not `"196"`. Scores are estimated ratings, not actual ratings.

In [42]:
user_id = 196
movies = pd.read_csv(
    r"u.item", sep="|", encoding="latin-1", header=None,
    usecols=[0, 1], names=["item_id", "title"],
)
movie_titles = movies.set_index("item_id")["title"]
full_trainset = surprise_data.build_full_trainset()
recommendation_model = SVD(n_factors=best_factor_count, random_state=42)
recommendation_model.fit(full_trainset)

rated_items = set(data.loc[data["user_id"] == user_id, "item_id"])
unseen_items = [item_id for item_id in data["item_id"].unique() if item_id not in rated_items]
recommendation_predictions = [
    recommendation_model.predict(user_id, item_id) for item_id in unseen_items
]
top_five_predictions = sorted(
    recommendation_predictions, key=lambda prediction: (-prediction.est, prediction.iid)
)[:5]
top_five = pd.DataFrame([
    {"item_id": prediction.iid, "estimated_rating": prediction.est}
    for prediction in top_five_predictions
])
top_five.insert(1, "title", top_five["item_id"].map(movie_titles))
assert top_five["title"].notna().all(), "Missing movie titles in u.item"
assert len(top_five) == 5 and top_five["item_id"].is_unique
assert rated_items.isdisjoint(top_five["item_id"])
display(top_five)

,item_id,title,estimated_rating
0,408,"Close Shave, A (1995)",4.521738
1,64,"Shawshank Redemption, The (1994)",4.484879
2,318,Schindler's List (1993),4.475977
3,483,Casablanca (1942),4.433455
4,169,"Wrong Trousers, The (1993)",4.425081


# Optional Task 3:
Use GridSearchCV or RandomizedSearchCV to find the optimal values for n_factors, lr_all(learning rate - e.g., 0.005, 0.007), and reg_all(regularisation - e.g., 0.02, 0.1). Your target is to achieve an RMSE lower than 0.94.

In [38]:
training_ratings = pd.DataFrame([
    (trainset.to_raw_uid(user), trainset.to_raw_iid(item), rating)
    for user, item, rating in trainset.all_ratings()
], columns=["user_id", "item_id", "rating"])
tuning_data = Dataset.load_from_df(training_ratings, Reader(rating_scale=(1, 5)))

param_grid = {
    "n_factors": [100],
    "lr_all": [0.005, 0.007, 0.01],
    "reg_all": [0.1],
    "n_epochs": [20, 30, 40, 50],
    "random_state": [42],
}

cv = KFold(n_splits=3, random_state=42, shuffle=True)
grid_search = GridSearchCV(SVD, param_grid, measures=["rmse"], cv=cv, n_jobs=1)
grid_search.fit(tuning_data)

grid_results = pd.DataFrame(grid_search.cv_results)
display(grid_results[["params", "mean_test_rmse"]].sort_values("mean_test_rmse"))
print("Best parameters:", grid_search.best_params["rmse"])
print(f"Best cross-validation RMSE: {grid_search.best_score['rmse']:.4f}")

tuned_model = grid_search.best_estimator["rmse"]
tuned_model.fit(trainset)
tuned_predictions = tuned_model.test(testset)
tuned_rmse = accuracy.rmse(tuned_predictions, verbose=False)
print(f"Held-out test RMSE: {tuned_rmse:.4f}")
print("Target reached." if tuned_rmse < 0.94 else "Target not reached: test RMSE must be below 0.94.")

,params,mean_test_rmse
10,"{'n_factors': 100, 'lr_all': 0.01, 'reg_all': ...",0.935880
7,"{'n_factors': 100, 'lr_all': 0.007, 'reg_all':...",0.935892
11,"{'n_factors': 100, 'lr_all': 0.01, 'reg_all': ...",0.936187
9,"{'n_factors': 100, 'lr_all': 0.01, 'reg_all': ...",0.936617
6,"{'n_factors': 100, 'lr_all': 0.007, 'reg_all':...",0.937039
3,"{'n_factors': 100, 'lr_all': 0.005, 'reg_all':...",0.938175
5,"{'n_factors': 100, 'lr_all': 0.007, 'reg_all':...",0.940699
2,"{'n_factors': 100, 'lr_all': 0.005, 'reg_all':...",0.941431
8,"{'n_factors': 100, 'lr_all': 0.01, 'reg_all': ...",0.941659
1,"{'n_factors': 100, 'lr_all': 0.005, 'reg_all':...",0.945976


Best parameters: {'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.1, 'n_epochs': 40, 'random_state': 42}
Best cross-validation RMSE: 0.9359
Held-out test RMSE: 0.9160
Target reached.


# Optional Task 4:
Item-Item vs User-User:
- Run 2 KNNBasic models: one with user_based=True (User-User CF) and one with user_based=False(Item-Item CF).
- Which approach is more accurate(RMSE) for the MovieLens dataset and which one is faster?

KNNBasic documentation: https://surprise.readthedocs.io/en/stable/prediction_algorithms.html

In [39]:
def evaluate_knn(sim_options):
    model = KNNBasic(k=40, min_k=1, sim_options=sim_options, verbose=False)
    start = perf_counter()
    model.fit(trainset)
    fit_seconds = perf_counter() - start
    start = perf_counter()
    predictions = model.test(testset)
    predict_seconds = perf_counter() - start
    return {
        "RMSE": accuracy.rmse(predictions, verbose=False),
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        "total_seconds": fit_seconds + predict_seconds,
    }

knn_results = []
for user_based in [True, False]:
    result = evaluate_knn({"name": "msd", "user_based": user_based})
    result["approach"] = "User-User" if user_based else "Item-Item"
    knn_results.append(result)

knn_results = pd.DataFrame(knn_results)
display(knn_results[["approach", "RMSE", "fit_seconds", "predict_seconds", "total_seconds"]])
most_accurate = knn_results.loc[knn_results["RMSE"].idxmin(), "approach"]
fastest = knn_results.loc[knn_results["total_seconds"].idxmin(), "approach"]
print(f"Lowest RMSE: {most_accurate}")
print(f"Fastest fit + prediction in this run: {fastest}")

,approach,RMSE,fit_seconds,predict_seconds,total_seconds
0,User-User,0.985369,1.690207,13.853425,15.543632
1,Item-Item,0.978571,1.847909,12.302310,14.150219


Lowest RMSE: Item-Item
Fastest fit + prediction in this run: Item-Item


# Optional Task 5:
Run the KNNBasic model using 3 different similarity metrics available in Surprise:
- name='cosine'
- name='pearson'(Pearson correlation coeficient)
- name='msd'(Mean Squared Difference)
  
Which metric yields the best RMSE?

In [40]:
similarity_results = []
for similarity in ["cosine", "pearson", "msd"]:
    if similarity == "msd":
        result = knn_results.loc[
            knn_results["approach"] == "User-User",
            ["RMSE", "fit_seconds", "predict_seconds", "total_seconds"],
        ].iloc[0].to_dict()
    else:
        result = evaluate_knn({"name": similarity, "user_based": True})
    result["similarity"] = similarity
    similarity_results.append(result)

similarity_results = pd.DataFrame(similarity_results)
display(similarity_results[["similarity", "RMSE"]].sort_values("RMSE"))
best_similarity = similarity_results.loc[similarity_results["RMSE"].idxmin(), "similarity"]
print(f"Lowest RMSE for User-User KNNBasic: {best_similarity}")

,similarity,RMSE
2,msd,0.985369
1,pearson,1.018246
0,cosine,1.022276


Lowest RMSE for User-User KNNBasic: msd


# Compare Top-5 after tuning

Refit a separate recommendation model using the latest grid-search winner and all known ratings, then rank the same unseen movies for user 196. Keep the original Task 2 recommendations for comparison.

In [43]:
tuned_recommendation_model = SVD(**grid_search.best_params["rmse"])
tuned_recommendation_model.fit(surprise_data.build_full_trainset())
tuned_recommendation_predictions = [
    tuned_recommendation_model.predict(user_id, item_id) for item_id in unseen_items
]
tuned_top_five_predictions = sorted(
    tuned_recommendation_predictions, key=lambda prediction: (-prediction.est, prediction.iid)
)[:5]
tuned_top_five = pd.DataFrame([
    {"item_id": prediction.iid, "estimated_rating": prediction.est}
    for prediction in tuned_top_five_predictions
])
tuned_top_five.insert(1, "title", tuned_top_five["item_id"].map(movie_titles))
assert tuned_top_five["title"].notna().all(), "Missing movie titles in u.item"
assert len(tuned_top_five) == 5 and tuned_top_five["item_id"].is_unique
assert rated_items.isdisjoint(tuned_top_five["item_id"])

print(f"Original Top-5 for user {user_id}:")
display(top_five)
print("Tuned model parameters:", grid_search.best_params["rmse"])
print(f"Tuned Top-5 for user {user_id}:")
display(tuned_top_five)

Original Top-5 for user 196:


,item_id,title,estimated_rating
0,408,"Close Shave, A (1995)",4.521738
1,64,"Shawshank Redemption, The (1994)",4.484879
2,318,Schindler's List (1993),4.475977
3,483,Casablanca (1942),4.433455
4,169,"Wrong Trousers, The (1993)",4.425081


Tuned model parameters: {'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.1, 'n_epochs': 40, 'random_state': 42}
Tuned Top-5 for user 196:


,item_id,title,estimated_rating
0,1449,Pather Panchali (1955),4.955029
1,318,Schindler's List (1993),4.534321
2,64,"Shawshank Redemption, The (1994)",4.528222
3,427,To Kill a Mockingbird (1962),4.433904
4,603,Rear Window (1954),4.408654
